# Synapse parameterization of a circuit

Assign physiological synapse parameters &mdash; drawn from statistical model distributions &mdash;
to the synapses of a **circuit** using the `SynapseParameterization` task.

A parameterization is composed of four kinds of building block, wired together by reference:

- **Distributions** &mdash; the statistical distributions individual parameters are drawn from.
- **Synaptic models** &mdash; a `ExcitatoryTsodyksMarkramSynapticModel` groups one distribution per synapse
  parameter (conductance, decay time, number of vesicles, ...).
- **Neuron sets** &mdash; source / target populations used to select *which* synapses to parameterize.
- **Synapse model assigners** &mdash; map a synaptic model onto a subset of an edge population.

This notebook demonstrates **all three** assigner types:

| Assigner | Synapses it parameterizes |
| --- | --- |
| `AllPairsSynapticModelAssigner` | every synapse in an edge population |
| `InterNeuronSetSynapticModelAssigner` | synapses between a source and a target neuron set |
| `PresynapticNeuronSetSynapticModelAssigner` | synapses *from* a source neuron set |

Assigners that share an edge population are applied in insertion order on top of a default
parameterization, so later (more specific) assigners refine earlier (broader) ones.

In [5]:
import shutil
from pathlib import Path

import numpy as np
import obi_one as obi
from entitysdk import Client, models
from obi_auth import get_token
from obi_notebook import get_projects

## Connect to the platform

The circuit used below lives in the **staging** environment.

In [8]:
environment = "staging"
token = get_token(environment=environment)
project_context = get_projects.get_projects(token=token, env=environment)

client = Client(project_context=project_context, token_manager=token, environment=environment)

Dropdown(description='Select:', options=(('My first project', {'id': '7a411785-6895-4839-aaa2-d9f76e09875a', '…

In [9]:
res = client.get_entity(entity_id="90351197-fed7-44a9-bb96-86a575a47538", entity_type=models.CellMorphology)

In [15]:
[f"{a.label}: {a.content_type}" for a in res.assets]

['morphology: application/swc',
 'morphology: application/asc',
 'morphology: application/x-hdf5',
 'morphology_with_spines: application/x-hdf5']

In [21]:
asset = client.fetch_assets(
    res,
    selection={"content_type": "application/x-hdf5"},
    output_path="./TMP/morph.h5",
).one()

IteratorResultError: There are more than one items.

In [6]:
res = client.get_entity(entity_id="e64f1432-ea7b-44b3-97d1-83140fb7b905", entity_type=models.Circuit)

In [7]:
res

Circuit(assets=[Asset(id=UUID('170bf519-a2f8-4aec-b160-3549cbc396b2'), creation_date=None, update_date=None, created_by=None, updated_by=None, path='sonata_circuit', full_path='private/e6030ed8-a589-4be2-80a6-f975406eb1f6/a66327e7-f5ca-4aee-8b6c-1ca3d3ab1a72/assets/circuit/e64f1432-ea7b-44b3-97d1-83140fb7b905/sonata_circuit', storage_type=<StorageType.aws_s3_internal: 'aws_s3_internal'>, is_directory=True, content_type=<ContentType.application_vnd_directory: 'application/vnd.directory'>, size=-1, sha256_digest=None, status=<AssetStatus.created: 'created'>, meta={}, label=<AssetLabel.sonata_circuit: 'sonata_circuit'>), Asset(id=UUID('43f731f7-9bf7-4441-a855-25f409d4c579'), creation_date=None, update_date=None, created_by=None, updated_by=None, path='circuit.tar.gz', full_path='private/e6030ed8-a589-4be2-80a6-f975406eb1f6/a66327e7-f5ca-4aee-8b6c-1ca3d3ab1a72/assets/circuit/e64f1432-ea7b-44b3-97d1-83140fb7b905/circuit.tar.gz', storage_type=<StorageType.aws_s3_internal: 'aws_s3_internal'>,

## Select the circuit

We use a public circuit in staging. `CircuitFromID` works with any circuit (it does not require
the "single" scale that the `MEModelWithSynapsesCircuitFromID` wrapper does).

In [ ]:
# Public circuit in staging
circuit_id = "8296b057-a0a0-41dc-86de-f64f6b8c0936"
circuit_from_id = obi.CircuitFromID(id_str=circuit_id)

# Staged circuits and the parameterized output are written here (outside the repo)
output_root = Path("../../../../../../obi-output/synapse_parameterization")
coordinate_output_root = output_root / "parameterized_circuit"

## Inspect the circuit

Stage the circuit once (into the entity cache the task will reuse) and look at its edge
populations and node sets to decide what to parameterize.

In [ ]:
circuit_entity = circuit_from_id.entity(db_client=client)
print(f"Circuit: {circuit_entity.name}  (id {circuit_entity.id}, scale {circuit_entity.scale})")

# Stage into the same entity cache the task reuses, so the circuit is only downloaded once
cache_dir = output_root.resolve() / "entity_cache" / "sonata_circuit" / str(circuit_entity.id)
circuit = circuit_from_id.stage_circuit(db_client=client, dest_dir=cache_dir, entity_cache=True)
sonata = circuit.sonata_circuit

print("\nEdge populations:")
for ep_name in sonata.edges.population_names:
    ep = sonata.edges[ep_name]
    print(f"  {ep_name}: {ep.source.name} -> {ep.target.name}  ({ep.size} synapses)")

# Parameterize the default intrinsic edge population (biophysical -> biophysical).
# Fall back gracefully if the default is ambiguous; you can also set this manually.
try:
    edge_population_name = circuit.default_edge_population_name
except ValueError:
    edge_population_name = None
if edge_population_name is None:
    intrinsic = [
        name
        for name in sonata.edges.population_names
        if sonata.edges[name].source.name == sonata.edges[name].target.name
    ]
    edge_population_name = (intrinsic or list(sonata.edges.population_names))[0]
ep = sonata.edges[edge_population_name]
print(
    f"\nParameterizing edge population '{edge_population_name}' "
    f"({ep.source.name} -> {ep.target.name}, {ep.size} synapses)"
)

# The property neuron sets below select excitatory / inhibitory neurons by the `synapse_class`
# property of the edge population's source population.
source_population = ep.source.name
source_node_properties = set(sonata.nodes[source_population].property_names)
print(
    f"'synapse_class' available in source population '{source_population}':",
    "synapse_class" in source_node_properties,
)

## Build the configuration

Start from an empty config and add blocks to it. `config.add(block, name=...)` stores the block in
the appropriate dictionary and creates a reference (`block.ref`) that other blocks can point to.

In [ ]:
config = obi.SynapseParameterizationSingleConfig.empty_config()

config.set(
    obi.Info(
        campaign_name="Synapse parameterization",
        campaign_description=f"Physiological parameterization of circuit {circuit_entity.name}",
    ),
    name="info",
)
config.set(config.Initialize(circuit=circuit_from_id), name="initialize")

### Synaptic models and their distributions

Each `ExcitatoryTsodyksMarkramSynapticModel` needs one distribution per synapse parameter. The helper below
adds the distributions to the config and wires the model to reference them. The default values
follow the platform's standard Tsodyks-Markram model; here the excitatory and inhibitory models
differ only in their conductance distribution (illustrative values).

In [ ]:
exc_gamma_dist = obi.GammaDistribution(shape=4.0, scale=0.25)
config.add(exc_gamma_dist, "Excitatory conductance distribution")

inh_gamma_dist = obi.GammaDistribution(shape=8.0, scale=0.25)
config.add(inh_gamma_dist, "Inhibitory conductance distribution")


# Excitatory model
excitatory_model = obi.ExcitatoryTsodyksMarkramSynapticModel(
    conductance_distribution=exc_gamma_dist.ref
)

# Inhibitory model (larger conductance, inhibitory synapse type id)
inhibitory_model = obi.InhibitoryTsodyksMarkramSynapticModel(
    conductance_distribution=inh_gamma_dist.ref
)
config.add(excitatory_model, "Excitatory synaptic model")
config.add(inhibitory_model, "Inhibitory synaptic model")

print("Synaptic models:", list(config.synaptic_models.keys()))
print("Distributions:", len(config.distributions))

### Neuron sets

The inter- and presynaptic assigners select synapses by the neuron set of the pre/post-synaptic
neurons. Excitatory and inhibitory neurons are selected with `BiophysicalPopulationPropertyNeuronSet`,
which resolves a single biophysical population by neuron properties &mdash; here the `synapse_class`
(`EXC` / `INH`). The population is the edge population's source population, because the assigners look
up neuron IDs under that population name. We fall back to "all neurons" if the source population has
no `synapse_class` property.

In [ ]:
all_neurons = obi.AllBiophysicalNeurons()
config.add(all_neurons, name="AllNeurons")

# Excitatory / inhibitory neurons selected by their `synapse_class` property (EXC / INH), resolved
# within the source population of the parameterized edge population. The presynaptic and inter
# assigners look up neuron IDs by population name, so `population` must be the edge population's
# source population.
if "synapse_class" in source_node_properties:
    excitatory_neurons = obi.BiophysicalPopulationPropertyNeuronSet(
        population=source_population,
        property_filter=obi.NeuronPropertyFilter(filter_dict={"synapse_class": ["EXC"]}),
    )
    config.add(excitatory_neurons, name="ExcitatoryNeurons")

    inhibitory_neurons = obi.BiophysicalPopulationPropertyNeuronSet(
        population=source_population,
        property_filter=obi.NeuronPropertyFilter(filter_dict={"synapse_class": ["INH"]}),
    )
    config.add(inhibitory_neurons, name="InhibitoryNeurons")
else:
    excitatory_neurons = all_neurons
    inhibitory_neurons = all_neurons
    print("No 'synapse_class' property - using all neurons for the excitatory/inhibitory sets.")

### Synapse model assigners

All three assigner types target the same edge population, so they are layered in the order they are
added: a broad baseline first, then progressively more specific refinements.

1. **`AllPairsSynapticModelAssigner`** &mdash; baseline: every synapse gets the excitatory model.
2. **`InterNeuronSetSynapticModelAssigner`** &mdash; synapses from *inhibitory* to *all* neurons get
   the inhibitory model.
3. **`PresynapticNeuronSetSynapticModelAssigner`** &mdash; synapses *from excitatory* neurons get the
   excitatory model.

In [ ]:
# 1. Baseline: assign the excitatory model to EVERY synapse in the edge population
all_pairs_assigner = obi.AllPairsSynapticModelAssigner(
    edge_population_name=edge_population_name,
    synaptic_model=excitatory_model.ref,
)
config.add(all_pairs_assigner, name="all_pairs")

# 2. Refine: inhibitory -> all synapses get the inhibitory model
inter_assigner = obi.InterNeuronSetSynapticModelAssigner(
    edge_population_name=edge_population_name,
    synaptic_model=inhibitory_model.ref,
    source_neuron_set=inhibitory_neurons.ref,
    targeted_neuron_set=all_neurons.ref,
)
config.add(inter_assigner, name="inhibitory_pathways")

# 3. Refine: synapses FROM excitatory neurons get the excitatory model
presyn_assigner = obi.PresynapticNeuronSetSynapticModelAssigner(
    edge_population_name=edge_population_name,
    synaptic_model=excitatory_model.ref,
    source_neuron_set=excitatory_neurons.ref,
)
config.add(presyn_assigner, name="excitatory_pathways")

print("Assigners:", list(config.synapse_model_assigners.keys()))

## Finalize and run the task

Resolving the block references links every assigner to its synaptic model and neuron sets, and
every synaptic model to its distributions.

Running the task stages the circuit, writes the drawn parameters into the SONATA edge files, and
finally performs a **dry-run registration** of the parameterized circuit as a derivation of the
original (validating the metadata against the platform without persisting anything).

In [ ]:
# Resolve references: assigner -> model / neuron sets, and model -> distributions
config.fill_block_references_and_names()

# Output locations for the staged + parameterized circuit
config.scan_output_root = output_root
config.coordinate_output_root = coordinate_output_root

task = obi.SynapseParameterizationTask(config=config)

In [ ]:
# The task copies the staged circuit into a fresh output directory; clear a previous run's output
if coordinate_output_root.exists():
    shutil.rmtree(coordinate_output_root)

task.execute(db_client=client, entity_cache=True)
print("Done. Parameterized circuit written to:", coordinate_output_root.resolve())

## Inspect the parameterized synapses

Open the parameterized circuit and check that the physiological properties were written. The
`syn_type_id` counts reveal the layering: synapses keep the excitatory or inhibitory model id
depending on which assigner last touched them.

In [ ]:
parameterized = obi.Circuit(
    name="parameterized_circuit",
    path=str(coordinate_output_root / "circuit_config.json"),
).sonata_circuit
edges = parameterized.edges[edge_population_name]

physiology_props = [
    p for p in obi.ExcitatoryTsodyksMarkramSynapticModel.parameter_names() if p in edges.property_names
]
print(f"{edge_population_name}: {edges.size} synapses")
print("Parameterized properties:", physiology_props)

# Sample synapses for a quick look at the assigned distributions
ids = edges.ids()
if len(ids) > 50_000:
    ids = np.sort(np.random.default_rng(0).choice(ids, 50_000, replace=False))
syn_table = edges.get(ids, properties=physiology_props)

print("\nsyn_type_id counts (excitatory vs inhibitory model layering):")
print(syn_table["syn_type_id"].value_counts())

syn_table.head()

In [ ]:
syn_table.drop(columns=["syn_type_id"]).hist(figsize=(12, 8))